# 05 — Come fa il classificatore a feature ad andare così bene? (Versione 2)

Analisi del TF-IDF + regressione logistica sul dataset della Versione 2, con le **stesse 304 testate valutate** e gli
stessi **fold per gruppo**. Per velocità il modello di riferimento si addestra **solo sulle testate valutate**, con
C = 1 e classi bilanciate: è il controllo `feature_tuned_tetto5` della coda (F1 0,388), praticamente uguale al migliore
(`feature`, 0,389). La testata prende la **media delle probabilità** dei suoi articoli.

1. **Il dataset a vista**: lunghezze, stile, date e metadati NewsGuard per livello.
2. **Il modello di riferimento** e le sue predizioni.
3. **Le parole che pesano di più** per ogni livello: sono nomi propri? stanno in molte testate o in poche?
4. **Le parole che decidono** il voto di ogni testata di test.
5. **Ablazioni**: lo stesso modello su testo modificato (senza nomi propri, solo parole funzione, senza parole funzione,
   solo l'inizio dell'articolo, senza le parole più forti, solo unigrammi).
6. **Altri segnali**: solo stile (lunghezze, punteggiatura, pronomi…), solo le date, solo i metadati NewsGuard
   (orientamento, paese, tipo di testata: non è testo, è un riferimento per i confondenti).
7. **Orientamento politico**: il TF-IDF riconosce l'orientamento? Quanto va bene dentro ogni orientamento?
8. **Quanti articoli servono** per giudicare una testata.
9. **Gli errori**: dove sbaglia.

Ogni sezione è protetta: se una fallisce, lo scrive nel log e le altre vanno avanti. Tutti i risultati in
`Results/2-classificazione/analisi_feature/`. Sul server: `papermill -k johnvenv 05_analisi_feature.ipynb ...`.

In [1]:
import json
import re
import time
import traceback
from collections import Counter

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler

from comune import CLASSI, RESULTS, SEED, carica, registratore

In [2]:
PROVA = False
N_JOBS = 40
C = 1.0

In [3]:
OUT = (RESULTS / "prova" if PROVA else RESULTS) / "analisi_feature"
OUT.mkdir(parents=True, exist_ok=True)
registra = registratore(OUT / "avanzamento.log")
LIV = ["l1", "l2", "l3", "l4", "l5"]
T0 = time.time()

art = carica(PROVA, solo_valutazione=True).reset_index(drop=True)
art["data"] = pd.to_datetime(art.data, errors="coerce")
y = art.label.to_numpy()
fold = art.fold_gruppo.to_numpy()
FOLDS = sorted(np.unique(fold))
ng = pd.read_csv("../../Dataset/newsguard.csv", low_memory=False)
ng = ng.assign(d=ng.Domain.str.lower()).drop_duplicates("d").set_index("d")
ff = pd.read_csv("../Dataset/versioni/fold_finale.csv").set_index("domain")
testate = art.groupby("domain").agg(label=("label", "first"), gruppo=("gruppo", "first"), fold=("fold_gruppo", "first"),
                                    articoli=("article_id", "size")).reset_index()
testate["in_gruppo"] = testate.domain.map(ff.testate_valutate_nel_gruppo).fillna(1).to_numpy() > 1
for c in ["Country", "Orientation", "Opinion/Advocacy Journalism", "Type of Content", "Scope of Coverage", "Topics", "Paywall"]:
    testate[c] = testate.domain.map(ng[c]).fillna("(vuoto)").astype(str)
registra(f"=== analisi del feature: {len(art)} articoli, {len(testate)} testate, fold {FOLDS} ===")


def sezione(nome):
    # decoratore: una sezione che fallisce non ferma le altre
    def avvolgi(f):
        t = time.time(); registra(f"--- {nome}")
        try:
            r = f()
            registra(f"    fatta in {(time.time() - t) / 60:.1f} min")
            return r
        except Exception:
            registra(f"!!! {nome} FALLITA:\n{traceback.format_exc()}")
    return avvolgi


def per_testata(idx, P):
    # dalle probabilità per articolo (righe idx di art) alla testata: media delle probabilità
    d = pd.DataFrame(P, columns=CLASSI); d["domain"] = art.domain.to_numpy()[idx]; d["y"] = y[idx]
    m = d.groupby("domain").agg({**{c: "mean" for c in CLASSI}, "y": "first"})
    return pd.DataFrame({"y_true": m.y, "y_pred": m[CLASSI].to_numpy().argmax(1)}, index=m.index)


def punteggi(t):
    return {"F1 macro": f1_score(t.y_true, t.y_pred, average="macro", labels=range(5), zero_division=0),
            "accuratezza": accuracy_score(t.y_true, t.y_pred),
            "binario": accuracy_score(t.y_true >= 2, t.y_pred >= 2)}

2026-10-20 11:01:49  === analisi del feature: 44012 articoli, 304 testate, fold [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)] ===


## 1. Il dataset a vista

Per livello: quanto sono lunghi gli articoli, quanta punteggiatura, citazioni, numeri, pronomi; quando sono stati
pubblicati; che cosa dice NewsGuard delle testate (paese, orientamento, tipo di contenuto).

In [4]:
PRIMA = {"i", "me", "my", "we", "us", "our"}
SECONDA = {"you", "your"}


def stile(t):
    parole = t.split()
    n = max(len(parole), 1)
    frasi = [f for f in re.split(r"[.!?]+\s", t) if f.strip()]
    lett = sum(c.isalpha() for c in t) or 1
    minus = [w.lower().strip(".,;:!?\"'“”‘’()") for w in parole]
    return {"parole": len(parole), "parole per frase": n / max(len(frasi), 1),
            "maiuscole %": 100 * sum(c.isupper() for c in t) / lett,
            "! ogni 1000 parole": 1000 * t.count("!") / n, "? ogni 1000 parole": 1000 * t.count("?") / n,
            "virgolette ogni 1000 parole": 1000 * sum(t.count(q) for q in ['"', "“", "”"]) / n,
            "numeri ogni 1000 parole": 1000 * len(re.findall(r"\d+", t)) / n,
            "io/noi ogni 1000 parole": 1000 * sum(w in PRIMA for w in minus) / n,
            "tu/voi ogni 1000 parole": 1000 * sum(w in SECONDA for w in minus) / n,
            "parole lunghe %": 100 * sum(len(w) >= 8 for w in minus) / n,
            "parole diverse % (prime 300)": 100 * len(set(minus[:300])) / max(min(n, 300), 1),
            "parentesi ogni 1000 parole": 1000 * t.count("(") / n}


STILE = pd.DataFrame(Parallel(n_jobs=N_JOBS, batch_size=500)(delayed(stile)(t) for t in art.text))
STILE["livello"] = art.label.map(dict(enumerate(LIV))).to_numpy()


@sezione("1. dataset a vista")
def _():
    tab = STILE.groupby("livello").median().T.round(1)
    tab.to_csv(OUT / "1_stile_per_livello_mediane.csv")
    display(tab)
    giorni = (art.data - pd.Timestamp("2025-01-01")).dt.days
    date = pd.DataFrame({"livello": STILE.livello, "giorni": giorni, "anno": art.data.dt.year, "mese": art.data.dt.to_period("M").astype(str)})
    dt = date.groupby("livello").agg(mediana_data=("giorni", lambda g: (pd.Timestamp("2025-01-01") + pd.Timedelta(days=float(g.median()))).date()),
                                     quota_2025=("anno", lambda a: float((a == 2025).mean())),
                                     senza_data=("giorni", lambda g: float(g.isna().mean())))
    dt.to_csv(OUT / "1_date_per_livello.csv"); display(dt)
    for c in ["Country", "Orientation", "Opinion/Advocacy Journalism", "Type of Content", "Scope of Coverage", "Paywall"]:
        x = pd.crosstab(testate[c], testate.label.map(dict(enumerate(LIV))), normalize="columns").round(2)
        x = x[x.max(axis=1) >= 0.05]
        x.to_csv(OUT / f"1_metadati_{re.sub(r'[^a-z]+', '_', c.lower())}.csv")
        print(f"\n{c} (quota di testate per livello)"); display(x)
    e = testate.assign(argomento=testate.Topics.str.split(",")).explode("argomento").reset_index(drop=True)
    a = pd.crosstab(e.argomento.str.strip(), e.label.map(dict(enumerate(LIV))))
    a = (a / testate.label.value_counts().sort_index().set_axis(LIV)).round(2)
    a = a[a.max(axis=1) >= 0.1]; a.to_csv(OUT / "1_argomenti_newsguard.csv"); print("\nargomenti NewsGuard (quota di testate)"); display(a)

2026-10-20 11:01:52  --- 1. dataset a vista


livello,l1,l2,l3,l4,l5
parole,572.5,572.0,602.0,572.0,743.5
parole per frase,21.0,20.1,20.6,22.5,22.5
maiuscole %,3.8,4.3,3.9,3.7,3.8
! ogni 1000 parole,0.0,0.0,0.0,0.0,0.0
? ogni 1000 parole,0.0,0.0,0.0,0.0,0.0
virgolette ogni 1000 parole,13.7,18.9,14.1,15.6,18.2
numeri ogni 1000 parole,14.1,16.4,12.7,13.7,14.5
io/noi ogni 1000 parole,4.2,4.9,8.2,6.0,5.0
tu/voi ogni 1000 parole,0.0,0.0,1.5,0.0,0.0
parole lunghe %,20.0,19.3,17.2,18.9,18.8


,mediana_data,quota_2025,senza_data
livello,,,
l1,2026-09-09,0.143123,0.0
l2,2026-09-29,0.027805,0.0
l3,2026-10-01,0.095379,0.0
l4,2026-10-01,0.065084,0.0
l5,2026-10-01,0.032740,0.0



Country (quota di testate per livello)


label,l1,l2,l3,l4,l5
Country,,,,,
AU,0.00,0.00,0.00,0.08,0.03
CA,0.03,0.02,0.00,0.04,0.14
GB,0.04,0.03,0.17,0.20,0.11
US,0.92,0.94,0.83,0.65,0.72


...


Paywall (quota di testate per livello)


label,l1,l2,l3,l4,l5
Paywall,,,,,
No,0.93,0.97,0.94,0.83,0.78
Yes,0.04,0.02,0.06,0.17,0.22



argomenti NewsGuard (quota di testate)


label,l1,l2,l3,l4,l5
argomento,,,,,
Conspiracy theories or hoaxes,0.56,0.08,0.11,0.01,0.00
General / Local News,0.01,0.12,0.00,0.08,0.06
Health or medical information,0.32,0.08,0.14,0.12,0.17
Personal finance news or advice,0.03,0.00,0.11,0.12,0.00
Political news or commentary,1.00,1.00,1.00,0.99,1.00


2026-10-20 11:01:52      fatta in 0.0 min


## 2. Il modello di riferimento e le ablazioni

Lo stesso TF-IDF (1–2 grammi, min_df 5, max_df 0,9, sublinear, 100.000 parole) + regressione logistica (C = 1, classi
bilanciate), addestrato fold per fold sul solo training. Le ablazioni cambiano **solo il testo** (o il vocabolario),
tutto il resto è uguale:

| variante | cosa cambia |
|---|---|
| `riferimento` | niente |
| `senza nomi propri` | via le parole con la maiuscola che non aprono una frase (nomi di persone, luoghi, testate, sigle) |
| `solo parole funzione` | restano solo le parole funzione inglesi (the, of, and, but, would…): lo stile, senza il contenuto |
| `senza parole funzione` | via le parole funzione: il contenuto, senza lo stile |
| `primi 100 parole`, `primi 300 parole` | solo l'inizio dell'articolo |
| `senza le 1000 più forti` | via le 1000 parole con i coefficienti più grandi (scelte sul training del fold), e si riaddestra |
| `solo unigrammi` | niente coppie di parole |

In [5]:
BASE = dict(ngram_range=(1, 2), min_df=5, max_df=0.9, sublinear_tf=True, max_features=100_000, strip_accents="unicode")
if PROVA:
    BASE["min_df"] = 2


def senza_nomi(t):
    righe = []
    for riga in t.split("\n"):
        fr = []
        for f in re.split(r"(?<=[.!?])\s+", riga):
            w = f.split()
            fr.append(" ".join(w[:1] + [x for x in w[1:] if not x[:1].isupper()]))
        righe.append(" ".join(fr))
    return "\n".join(righe)


def solo_funzione(t):
    return " ".join(w for w in re.findall(r"[a-z']+", t.lower()) if w in ENGLISH_STOP_WORDS)


def primi(n):
    return lambda t: " ".join(t.split()[:n])


TESTI = {"testo": art.text.to_numpy()}
for nome, f in [("senza_nomi", senza_nomi), ("solo_funzione", solo_funzione), ("primi100", primi(100)), ("primi300", primi(300))]:
    TESTI[nome] = np.array(Parallel(n_jobs=N_JOBS, batch_size=500)(delayed(f)(t) for t in art.text), dtype=object)
registra("testi preparati per le ablazioni")

VARIANTI = {
    "riferimento": ("testo", {}, 0),
    "senza nomi propri": ("senza_nomi", {}, 0),
    "solo parole funzione": ("solo_funzione", {"max_df": 1.0}, 0),
    "senza parole funzione": ("testo", {"stop_words": "english"}, 0),
    "primi 100 parole": ("primi100", {}, 0),
    "primi 300 parole": ("primi300", {}, 0),
    "senza le 1000 più forti": ("testo", {}, 1000),
    "solo unigrammi": ("testo", {"ngram_range": (1, 1)}, 0),
}


def un_fold(nome, testi, kw, togli, f):
    tr, te = np.where(fold != f)[0], np.where(fold == f)[0]
    v = TfidfVectorizer(**{**BASE, **kw})
    Xtr = v.fit_transform(testi[tr])
    clf = LogisticRegression(C=C, max_iter=2000, class_weight="balanced", random_state=SEED).fit(Xtr, y[tr])
    if togli:                                   # via le parole più forti (sul training) e si riaddestra
        forza = np.abs(clf.coef_).max(0)
        togli = min(togli, len(forza) // 2)
        tenute = np.sort(np.argsort(forza)[::-1][togli:])
        vocab = v.get_feature_names_out()[tenute]
        v = TfidfVectorizer(**{**BASE, **kw, "vocabulary": vocab, "max_features": None, "min_df": 1, "max_df": 1.0})
        Xtr = v.fit_transform(testi[tr])
        clf = LogisticRegression(C=C, max_iter=2000, class_weight="balanced", random_state=SEED).fit(Xtr, y[tr])
    P = np.zeros((len(te), 5)); P[:, clf.classes_] = clf.predict_proba(v.transform(testi[te]))
    modello = (v, clf) if nome == "riferimento" else None
    return nome, f, te, P, modello


t0 = time.time()
lavori = [(n, TESTI[k], kw, tg, f) for n, (k, kw, tg) in VARIANTI.items() for f in FOLDS]
risultati = Parallel(n_jobs=min(N_JOBS, len(lavori)), verbose=0)(delayed(un_fold)(*l) for l in lavori)
registra(f"{len(lavori)} addestramenti (varianti x fold) in {(time.time() - t0) / 60:.1f} min")
PROBA, MODELLI = {}, {}
for nome, f, te, P, modello in risultati:
    PROBA.setdefault(nome, np.zeros((len(art), 5)))[te] = P
    if modello:
        MODELLI[f] = modello
ABL = pd.DataFrame({n: punteggi(per_testata(np.arange(len(art)), P)) for n, P in PROBA.items()}).T.loc[list(VARIANTI)].round(3)
ABL.to_csv(OUT / "2_ablazioni.csv")
pd.DataFrame(PROBA["riferimento"], columns=CLASSI).assign(article_id=art.article_id, domain=art.domain, y_true=y, fold=fold).to_csv(
    OUT / "2_riferimento_articoli.csv", index=False)
registra("ablazioni:\n" + ABL.to_string())
ABL

2026-10-20 11:01:58  testi preparati per le ablazioni


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:1379: UserWarning: Upper case characters found in vocabulary while 'lowercase' is True. These entries will not be matched with any documents
  warnings.warn(


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:1379: UserWarning: Upper case characters found in vocabulary while 'lowercase' is True. These entries will not be matched with any documents
  warnings.warn(


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:1379: UserWarning: Upper case characters found in vocabulary while 'lowercase' is True. These entries will not be matched with any documents
  warnings.warn(


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:1379: UserWarning: Upper case characters found in vocabulary while 'lowercase' is True. These entries will not be matched with any documents
  warnings.warn(


2026-10-20 11:06:32  40 addestramenti (varianti x fold) in 4.6 min


2026-10-20 11:06:32  ablazioni:
                         F1 macro  accuratezza  binario
riferimento                 0.387        0.461    0.776
senza nomi propri           0.383        0.441    0.760
solo parole funzione        0.364        0.398    0.717
senza parole funzione       0.370        0.444    0.753
primi 100 parole            0.344        0.447    0.730
primi 300 parole            0.367        0.454    0.757
senza le 1000 più forti     0.377        0.444    0.753
solo unigrammi              0.387        0.457    0.773


,F1 macro,accuratezza,binario
riferimento,0.387,0.461,0.776
senza nomi propri,0.383,0.441,0.760
solo parole funzione,0.364,0.398,0.717
senza parole funzione,0.370,0.444,0.753
primi 100 parole,0.344,0.447,0.730
primi 300 parole,0.367,0.454,0.757
senza le 1000 più forti,0.377,0.444,0.753
solo unigrammi,0.387,0.457,0.773


In [6]:
RIF = per_testata(np.arange(len(art)), PROBA["riferimento"])
RIF = RIF.join(testate.set_index("domain"))
RIF["giusta"] = RIF.y_true == RIF.y_pred
cm = pd.DataFrame(confusion_matrix(RIF.y_true, RIF.y_pred, labels=range(5)), index=[f"vero {l}" for l in LIV], columns=[f"pred {l}" for l in LIV])
cm.to_csv(OUT / "2_confusione.csv")
display(cm)
fig, ax = plt.subplots(figsize=(7, 3.6))
ABL["F1 macro"][::-1].plot.barh(ax=ax, color=["#2a78d6" if i != len(ABL) - 1 else "#eb6834" for i in range(len(ABL))])
ax.axvline(ABL.loc["riferimento", "F1 macro"], color="k", ls="--", lw=1); ax.set_xlabel("F1 macro per testata")
ax.set_title("Ablazioni del TF-IDF (fold per gruppo, 304 testate)"); plt.tight_layout(); fig.savefig(OUT / "2_ablazioni.png", dpi=150); plt.close(fig)

,pred l1,pred l2,pred l3,pred l4,pred l5
vero l1,48,12,2,9,0
vero l2,22,27,2,12,2
vero l3,6,8,4,17,1
vero l4,3,17,8,54,14
vero l5,2,5,0,22,7


## 3. Le parole che pesano di più

Un modello addestrato su **tutte** le 304 testate, solo per leggere i coefficienti (non per valutare). Per le 40
parole più forti di ogni livello:

- in quante testate di quel livello compaiono, e in quante degli altri livelli;
- in quante testate e in quanti gruppi compaiono in tutto, e quanta parte delle occorrenze viene dalla testata che la
  usa di più (**concentrazione**: vicino a 1 = è la firma di una testata sola);
- quante volte compaiono con la **maiuscola** nel testo originale (un indizio di nome proprio; su un campione di
  10.000 articoli, per la prima parola dei bigrammi).

In [7]:
@sezione("3. parole più forti")
def _():
    global TOP
    v = TfidfVectorizer(**BASE); X = v.fit_transform(art.text)
    clf = LogisticRegression(C=C, max_iter=2000, class_weight="balanced", random_state=SEED).fit(X, y)
    nomi = v.get_feature_names_out()
    coef = clf.coef_ - clf.coef_.mean(0)              # quanto spinge verso il livello rispetto alla media dei livelli
    scelte = {(LIV[k], nomi[j]): coef[k, j] for k in range(len(clf.classes_)) for j in np.argsort(coef[k])[::-1][:40]}
    vocab = sorted({p for _, p in scelte})
    cv = CountVectorizer(vocabulary=vocab, ngram_range=(1, 2), strip_accents="unicode")
    B = cv.transform(art.text)
    per_t = pd.DataFrame(B.toarray(), columns=vocab).groupby(art.domain.to_numpy()).sum()
    lab_t = testate.set_index("domain").label.reindex(per_t.index)
    gru_t = testate.set_index("domain").gruppo.reindex(per_t.index)
    tutto = "\n".join(art.text.sample(min(10_000, len(art)), random_state=SEED).tolist())     # un campione basta per le maiuscole
    righe = []
    for (liv, p), w in scelte.items():
        k = LIV.index(liv); c = per_t[p]; pres = c > 0
        primo = p.split()[0]
        occ = re.findall(r"(?<![\w])" + re.escape(primo) + r"(?![\w])", tutto, flags=re.I)
        righe.append({"livello": liv, "parola": p, "peso": round(w, 3),
                      "testate del livello con la parola %": round(100 * (pres & (lab_t == k)).sum() / max((lab_t == k).sum(), 1), 1),
                      "testate degli altri livelli %": round(100 * (pres & (lab_t != k)).sum() / max((lab_t != k).sum(), 1), 1),
                      "testate": int(pres.sum()), "gruppi": int(gru_t[pres].nunique()),
                      "concentrazione": round(c.max() / max(c.sum(), 1), 2),
                      "maiuscola %": round(100 * sum(o[:1].isupper() for o in occ) / max(len(occ), 1))})
    TOP = pd.DataFrame(righe)
    TOP.to_csv(OUT / "3_parole_piu_forti.csv", index=False)
    for liv in LIV:
        print(f"\n=== {liv}"); display(TOP[TOP.livello == liv].drop(columns="livello").head(25).to_string(index=False))
    s = TOP.assign(nome_proprio=TOP["maiuscola %"] >= 50, firma=TOP.concentrazione >= 0.5).groupby("livello")[["nome_proprio", "firma"]].mean().round(2)
    s.to_csv(OUT / "3_quote_nomi_e_firme.csv"); print("\nquota delle 40 parole più forti che sono nomi propri (maiuscola >= 50%) o firme di una testata (concentrazione >= 0,5)"); display(s)

2026-10-20 11:06:33  --- 3. parole più forti



=== l1


'         parola  peso  testate del livello con la parola %  testate degli altri livelli %  testate  gruppi  concentrazione  maiuscola %\n            com 7.128                                 85.9                           80.7      249     176            0.14            1\npresident trump 5.058                                 88.7                           79.4      248     184            0.16           67\n   lifesitenews 4.271                                 11.3                            2.1       13      10            0.94           99\n       abortion 3.305                                 63.4                           52.4      167     123            0.38            5\n          video 3.027                                 94.4                           93.1      284     206            0.11           20\n           kiev 3.025                                 19.7                            5.6       27      25            0.44          100\n      louisiana 3.000                   


=== l2


'             parola  peso  testate del livello con la parola %  testate degli altri livelli %  testate  gruppi  concentrazione  maiuscola %\n               2026 4.351                                100.0                           97.5      298     214            0.13            0\n             alaska 4.012                                 67.7                           52.7      170     125            0.54          100\n                fox 3.895                                 81.5                           81.2      247     181            0.13          100\n           cannabis 3.623                                 20.0                           26.4       76      64            0.60           10\n       news digital 3.520                                 47.7                           25.1       91      60            0.14           66\n           redstate 3.416                                 24.6                            5.0       28      14            0.24           89\n     judicia

...


=== l5


'             parola  peso  testate del livello con la parola %  testate degli altri livelli %  testate  gruppi  concentrazione  maiuscola %\n              irish 5.497                                 63.9                           44.0      141     112            0.60          100\n              lgbtq 4.505                                 50.0                           44.8      138     106            0.45          100\npopular information 3.961                                  5.6                            2.6        9       7            0.94           14\n            defence 3.946                                 52.8                           47.4      146     108            0.17           26\n                 pa 3.941                                 47.2                           41.4      128      95            0.63          100\n      pressprogress 3.853                                  2.8                            0.0        1       1            1.00           97\n            


quota delle 40 parole più forti che sono nomi propri (maiuscola >= 50%) o firme di una testata (concentrazione >= 0,5)


,nome_proprio,firma
livello,,
l1,0.48,0.25
l2,0.75,0.18
l3,0.72,0.55
l4,0.68,0.60
l5,0.45,0.32


2026-10-20 11:14:18      fatta in 7.7 min


## 4. Le parole che decidono il voto di ogni testata

Per ogni testata di test, con il modello del suo fold: il vettore TF-IDF medio dei suoi articoli per il coefficiente di
ogni parola verso il livello predetto (rispetto alla media dei livelli). Le 10 parole che spingono di più. Poi, per
le testate indovinate di ogni livello, quali parole compaiono più spesso tra le prime 10: segnali **condivisi** tra
testate diverse o **firme** di una testata sola?

In [8]:
@sezione("4. contributi per testata")
def _():
    righe = []
    for f, (v, clf) in MODELLI.items():
        te = np.where(fold == f)[0]
        X = v.transform(art.text.to_numpy()[te]); nomi = v.get_feature_names_out()
        coef = np.zeros((5, X.shape[1])); coef[clf.classes_] = clf.coef_
        coef = coef - coef[clf.classes_].mean(0)
        d = art.domain.to_numpy()[te]
        for dom in np.unique(d):
            m = np.asarray(X[d == dom].mean(0)).ravel()
            pred = int(RIF.loc[dom, "y_pred"])
            contr = m * coef[pred]
            top = np.argsort(contr)[::-1][:10]
            righe.append({"domain": dom, "vero": LIV[int(RIF.loc[dom, "y_true"])], "predetto": LIV[pred], "giusta": bool(RIF.loc[dom, "giusta"]),
                          "parole decisive": " | ".join(nomi[top])})
    CONTR = pd.DataFrame(righe).sort_values(["vero", "giusta"])
    CONTR.to_csv(OUT / "4_parole_decisive_per_testata.csv", index=False)
    for liv in LIV:
        g = CONTR[(CONTR.vero == liv) & CONTR.giusta]
        conta = Counter(p for s in g["parole decisive"] for p in s.split(" | "))
        print(f"\n=== {liv}: {len(g)} testate indovinate; parole più frequenti tra le 10 decisive (in quante testate):")
        print(", ".join(f"{p} ({n})" for p, n in conta.most_common(25)))
        print(g.head(6)[["domain", "parole decisive"]].to_string(index=False))

2026-10-20 11:14:18  --- 4. contributi per testata



=== l1: 48 testate indovinate; parole più frequenti tra le 10 decisive (in quante testate):
media (28), president trump (27), that the (26), com (23), this (22), video (21), america (17), us (16), these (16), trump (12), while (12), the us (11), military (8), not (8), broader (7), stated (6), democrats (6), israel (6), why (6), specific (6), reported (5), it is (5), health (5), system (5), article (4)
                  domain                                                                                              parole decisive
americanpatriotdaily.com       president trump | trump | america | stated | democrats | primary | media | this | that the | president
        bestnewshere.com                          video | 26 | trump | intel | judy | military | powerful | update | bombshell | june
        nationalfile.com reported | israeli | military | defense | americans | israel | president trump | official | while | that the
        off-guardian.org                            media 

## 5. Altri segnali, senza le parole

Sempre per testata, fold per gruppo, regressione logistica bilanciata con C scelto a 1 (dati standardizzati):

- **solo stile**: le misure della sezione 1 (lunghezze, punteggiatura, maiuscole, pronomi…), media per testata;
- **solo date**: data mediana di pubblicazione, dispersione, quota di articoli del 2025;
- **solo metadati NewsGuard** (non è testo! serve a capire i confondenti): paese, orientamento, opinione/advocacy,
  tipo di contenuto, copertura, paywall, argomenti.

In [9]:
def lr_testate(X, nome):
    X = np.asarray(X, dtype=float); X = np.nan_to_num(X)
    yt = testate.label.to_numpy(); ft = testate.fold.to_numpy(); pred = np.zeros(len(testate), dtype=int)
    for f in FOLDS:
        tr, te = ft != f, ft == f
        sc = StandardScaler().fit(X[tr])
        clf = LogisticRegression(C=1.0, max_iter=5000, class_weight="balanced", random_state=SEED).fit(sc.transform(X[tr]), yt[tr])
        pred[te] = clf.predict(sc.transform(X[te]))
    r = punteggi(pd.DataFrame({"y_true": yt, "y_pred": pred}))
    registra(f"    {nome}: F1 {r['F1 macro']:.3f}, binario {r['binario']:.3f}")
    return r


@sezione("5. altri segnali")
def _():
    st = STILE.drop(columns="livello").groupby(art.domain.to_numpy()).mean().reindex(testate.domain)
    giorni = (art.data - pd.Timestamp("2025-01-01")).dt.days
    dt = pd.DataFrame({"g": giorni, "a": (art.data.dt.year == 2025).astype(float)}).groupby(art.domain.to_numpy()).agg(
        mediana=("g", "median"), dispersione=("g", "std"), q2025=("a", "mean")).reindex(testate.domain)
    meta = pd.get_dummies(testate[["Country", "Orientation", "Opinion/Advocacy Journalism", "Type of Content", "Scope of Coverage", "Paywall"]].astype(str))
    for a in ["Conspiracy", "Health", "General / Local", "Political", "Business", "Entertainment"]:
        meta[f"argomento {a}"] = testate.Topics.str.contains(a, regex=False)
    tab = pd.DataFrame({"solo stile (12 misure)": lr_testate(st, "solo stile"),
                        "solo date": lr_testate(dt, "solo date"),
                        "solo metadati NewsGuard (non testo)": lr_testate(meta, "solo metadati"),
                        "TF-IDF (riferimento)": punteggi(RIF)}).T.round(3)
    tab.to_csv(OUT / "5_altri_segnali.csv"); display(tab)

2026-10-20 11:15:22  --- 5. altri segnali


2026-10-20 11:15:22      solo stile: F1 0.296, binario 0.635


2026-10-20 11:15:22      solo date: F1 0.150, binario 0.543


2026-10-20 11:15:22      solo metadati: F1 0.339, binario 0.688


,F1 macro,accuratezza,binario
solo stile (12 misure),0.296,0.306,0.635
solo date,0.150,0.178,0.543
solo metadati NewsGuard (non testo),0.339,0.362,0.688
TF-IDF (riferimento),0.387,0.461,0.776


2026-10-20 11:15:22      fatta in 0.0 min


## 6. Orientamento politico

NewsGuard dà un orientamento solo ad alcune testate. (a) Il TF-IDF, con gli stessi fold per gruppo, sa riconoscere
l'orientamento dal testo? (b) Quanto va bene il TF-IDF sul livello **dentro** ogni orientamento? Se va bene solo dove
orientamento e livello coincidono (per esempio Far Right quasi tutte $l_1$), sta leggendo l'ideologia.

In [10]:
@sezione("6. orientamento")
def _():
    print("livello per orientamento (testate):")
    x = pd.crosstab(testate.Orientation, testate.label.map(dict(enumerate(LIV)))); display(x); x.to_csv(OUT / "6_orientamento_livello.csv")
    dentro = RIF.groupby("Orientation").apply(lambda g: pd.Series({"testate": len(g), "accuratezza": g.giusta.mean(),
                                                                   "maggioranza nel gruppo": g.y_true.value_counts(normalize=True).max()}))
    display(dentro.round(2)); dentro.to_csv(OUT / "6_accuratezza_per_orientamento.csv")
    con = testate[testate.Orientation != "(vuoto)"]
    if con.Orientation.nunique() < 2:
        return
    mappa = {o: i for i, o in enumerate(sorted(con.Orientation.unique()))}
    sel = art.domain.isin(con.domain).to_numpy()
    yo = art.domain.map(con.set_index("domain").Orientation).map(mappa).to_numpy()
    P = np.zeros((len(art), len(mappa)))

    def f_or(f):
        tr = np.where(sel & (fold != f))[0]; te = np.where(sel & (fold == f))[0]
        v = TfidfVectorizer(**BASE); clf = LogisticRegression(C=C, max_iter=2000, class_weight="balanced", random_state=SEED)
        clf.fit(v.fit_transform(art.text.to_numpy()[tr]), yo[tr].astype(int))
        Q = np.zeros((len(te), len(mappa))); Q[:, clf.classes_] = clf.predict_proba(v.transform(art.text.to_numpy()[te]))
        return te, Q
    for te, Q in Parallel(n_jobs=len(FOLDS))(delayed(f_or)(f) for f in FOLDS):
        P[te] = Q
    d = pd.DataFrame(P[sel]); d["domain"] = art.domain.to_numpy()[sel]
    m = d.groupby("domain").mean(); pred = m.to_numpy().argmax(1)
    vero = con.set_index("domain").Orientation.reindex(m.index).map(mappa).to_numpy()
    r = {"testate": len(m), "accuratezza orientamento": accuracy_score(vero, pred),
         "F1 macro orientamento": f1_score(vero, pred, average="macro"), "classe più frequente": pd.Series(vero).value_counts(normalize=True).max()}
    registra(f"    TF-IDF -> orientamento: {r}")
    pd.Series(r).to_csv(OUT / "6_tfidf_predice_orientamento.csv")
    display(pd.Series(r).round(3))

2026-10-20 11:15:22  --- 6. orientamento


livello per orientamento (testate):


label,l1,l2,l3,l4,l5
Orientation,,,,,
(vuoto),10,8,7,40,17
Far Left,0,4,1,8,3
Far Right,49,21,9,5,0
Slightly Left,1,5,4,23,11
Slightly Right,11,27,15,20,5


,testate,accuratezza,maggioranza nel gruppo
Orientation,,,
(vuoto),82.0,0.51,0.49
Far Left,16.0,0.31,0.50
Far Right,84.0,0.55,0.58
Slightly Left,44.0,0.36,0.52
Slightly Right,78.0,0.40,0.35


2026-10-20 11:17:27      TF-IDF -> orientamento: {'testate': 222, 'accuratezza orientamento': 0.5765765765765766, 'F1 macro orientamento': 0.4498815586036682, 'classe più frequente': np.float64(0.3783783783783784)}


testate                     222.000
accuratezza orientamento      0.577
F1 macro orientamento         0.450
classe più frequente          0.378
dtype: float64

2026-10-20 11:17:27      fatta in 2.1 min


## 7. Quanti articoli servono

Dalle probabilità del modello di riferimento: per ogni testata si pescano $k$ articoli a caso (20 ripetizioni) e si
fa la media delle probabilità. Se bastano pochi articoli, il segnale è forte in ogni articolo; se ne servono tanti, è
un segnale debole che si accumula.

In [11]:
@sezione("7. quanti articoli")
def _():
    rng = np.random.default_rng(SEED); P = PROBA["riferimento"]
    pos = art.groupby("domain").indices; righe = []
    for k in [1, 2, 3, 5, 10, 20, 50, 1000]:
        f1s, bins = [], []
        for _ in range(20 if k < 1000 else 1):
            pred, vero = [], []
            for d, idx in pos.items():
                s = rng.choice(idx, min(k, len(idx)), replace=False)
                pred.append(P[s].mean(0).argmax()); vero.append(y[idx[0]])
            t = pd.DataFrame({"y_true": vero, "y_pred": pred}); r = punteggi(t); f1s.append(r["F1 macro"]); bins.append(r["binario"])
        righe.append({"articoli per testata": "tutti" if k == 1000 else k, "F1 macro": np.mean(f1s), "binario": np.mean(bins)})
    cur = pd.DataFrame(righe).round(3); cur.to_csv(OUT / "7_quanti_articoli.csv", index=False); display(cur)

2026-10-20 11:17:27  --- 7. quanti articoli


,articoli per testata,F1 macro,binario
0,1,0.325,0.670
1,2,0.345,0.693
2,3,0.355,0.708
3,5,0.372,0.729
4,10,0.383,0.747
5,20,0.392,0.765
6,50,0.391,0.772
7,tutti,0.387,0.776


2026-10-20 11:17:28      fatta in 0.0 min


## 8. Dove sbaglia

Accuratezza per testata (modello di riferimento) secondo: livello, testata isolata o in un gruppo, numero di articoli,
paese, orientamento.

In [12]:
@sezione("8. errori")
def _():
    R = RIF.copy()
    R["livello"] = R.y_true.map(dict(enumerate(LIV)))
    R["articoli (fasce)"] = pd.cut(R.articoli, [0, 50, 100, 150, 1000], labels=["20-50", "51-100", "101-150", ">150"]).astype(str)
    R["paese"] = R.Country.where(R.Country.isin(["US", "GB", "CA", "AU"]), "altro")
    out = []
    for c in ["livello", "in_gruppo", "articoli (fasce)", "paese", "Orientation"]:
        g = R.groupby(c).agg(testate=("giusta", "size"), accuratezza=("giusta", "mean"),
                             errore_medio_livelli=("y_pred", lambda p: float(np.abs(p - R.loc[p.index, "y_true"]).mean())))
        g.index = [f"{c}: {i}" for i in g.index]; out.append(g)
    E = pd.concat(out).round(2); E.to_csv(OUT / "8_errori.csv"); display(E)
    R[["livello", "y_pred", "giusta", "in_gruppo", "articoli", "Country", "Orientation"]].to_csv(OUT / "8_testate.csv")
    registra(f"=== analisi finita in {(time.time() - T0) / 60:.1f} min ===")
    (OUT / "FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))

2026-10-20 11:17:28  --- 8. errori


,testate,accuratezza,errore_medio_livelli
livello: l1,71,0.68,0.61
livello: l2,65,0.42,0.83
livello: l3,36,0.11,1.08
livello: l4,96,0.56,0.68
livello: l5,36,0.19,1.25
in_gruppo: False,181,0.43,0.85
in_gruppo: True,123,0.51,0.75
articoli (fasce): 101-150,28,0.54,0.68
articoli (fasce): 20-50,45,0.49,0.84
articoli (fasce): 51-100,43,0.56,0.79


2026-10-20 11:17:29  === analisi finita in 15.7 min ===


2026-10-20 11:17:29      fatta in 0.0 min
